In [ ]:
import pandas as pd
import scipy as sc
import numpy as np
import skimage as sk
from glob import glob
import os
import napari
from PIL import Image

In [ ]:
def get_tissue_orientation_transforms(mask_points,tissue_array):
    """
    Determine the flip and/or rotation needed to orient the moving image
    to match the map, using the N and E cardinal point locations.

    Detection logic (image coordinates, y increases downward):
        North marker should be visually "above" East  → north_y < east_y
        East  marker should be visually "right" of North → east_x > north_x

    Flip cases:
        north_is_up  and east_is_right  → horizontal flip
        north_is_up  and east_is_left → none
        north_is_right and east_is_up  → CCW rotation
        north_is_right and east_is_down → CCW rotation + horizontal flip
        north_is_left and east_is_up → CW rotation + Vertical flip
        north_is_left and east_is_down → CW rotation
        north_is_down and east_is_left → vertical flip
        north_is_down and east_is_right → vertical + horizontal flip

    Parameters
    ----------
    mask_points  : dict with 'north' and 'east' as (x, y) pixel arrays — moving image
    moving_image: image array of moving image
    Returns flip transform matrix and rotation angle
    -------
    """
    FLIP_MATRICES = {
    "none":       [ 1,  0,  0,  1],
    "horizontal": [-1,  0,  0,  1],
    "vertical":   [ 1,  0,  0, -1],
    "both":       [-1,  0,  0, -1],
    }
    ROTATION_DICTIONARY = {
        'clockwise': -90,
        'counter-clockwise': 90
    }
    # --- Detect required flip from cardinal point geometry ---
    # Compare pixel coords directly for orientation — spacing does not
    # affect which direction is "up" or "right"
    #get moving image relative cardinal coordinates:
    north = (0, tissue_array.shape[1]//2)
    south = (tissue_array.shape[0]-1, tissue_array.shape[1]//2)
    west = (tissue_array.shape[0]//2, tissue_array.shape[1]-1)
    east = (tissue_array.shape[0]//2, 0)
    cardinals = {"up": north, "down": south, "left": east, "right": west}

    #which cardinal direction are the two reference points closest to?
    closest_north = min(cardinals, 
                  key=lambda d: np.hypot(mask_points['north'][0] - cardinals[d][0], 
                    mask_points['north'][1] - cardinals[d][1]))
    closest_east = min(cardinals, 
                  key=lambda d: np.hypot(mask_points['east'][0] - cardinals[d][0], 
                    mask_points['east'][1] - cardinals[d][1]))

    if closest_north == 'up' and closest_east == 'right':
        flip = 'horizontal'
        rotation = None
        flip_transform = FLIP_MATRICES[flip] if flip != None else None
        rotation_angle = ROTATION_DICTIONARY[rotation] if rotation != None else None
    elif closest_north == 'up' and closest_east == 'left':
        flip = None
        rotation = None
        flip_transform = FLIP_MATRICES[flip] if flip != None else None
        rotation_angle = ROTATION_DICTIONARY[rotation] if rotation != None else None
    elif closest_north == 'right' and closest_east == 'up':
        flip = None
        rotation = 'counter-clockwise'
        flip_transform = FLIP_MATRICES[flip] if flip != None else None
        rotation_angle = ROTATION_DICTIONARY[rotation] if rotation != None else None
    elif closest_north == 'left' and closest_east == 'up':
        flip = 'vertical'
        rotation = 'clockwise'
        flip_transform = FLIP_MATRICES[flip] if flip != None else None
        rotation_angle = ROTATION_DICTIONARY[rotation] if rotation != None else None
    elif closest_north == 'right' and closest_east == 'down':
        flip = 'horizontal'
        rotation = 'counter-clockwise'
        flip_transform = FLIP_MATRICES[flip] if flip != None else None
        rotation_angle = ROTATION_DICTIONARY[rotation] if rotation != None else None
    elif closest_north == 'left' and closest_east == 'down':
        flip = None
        rotation = 'clockwise'
        flip_transform = FLIP_MATRICES[flip] if flip != None else None
        rotation_angle = ROTATION_DICTIONARY[rotation] if rotation != None else None
    elif closest_north == 'down' and closest_east == 'right':
        flip = 'both'
        rotation = None
        flip_transform = FLIP_MATRICES[flip] if flip != None else None
        rotation_angle = ROTATION_DICTIONARY[rotation] if rotation != None else None
    elif closest_north == 'down' and closest_east == 'left':
        flip = 'vertical'
        rotation = None
        flip_transform = FLIP_MATRICES[flip] if flip != None else None
        rotation_angle = ROTATION_DICTIONARY[rotation] if rotation != None else None
    else:
        print('Orientation does not match conditions, check image')
    print(f'Detected Flip: {flip}')
    print(f'Detected Rotation: {rotation}') 
    return flip_transform, rotation_angle

In [ ]:
def detect_cardinal_point(arr, grey_value, name):
    """
    Detect the centroid of a cardinal point marker by its grey value.
    
    Returns (x, y) in pixel coordinates, or None if not found.
    """
    
    # Create binary mask for this grey value
    mask = (arr * (arr == int(grey_value))).astype(int)
        
    if not mask.any():
        print(f"  WARNING: No pixels found for grey value {grey_value} in {name}")
        return None
    
    # Label connected components and get centroid
    labeled_img = sk.measure.label(mask)
    props = sk.measure.regionprops(labeled_img)
    cy,cx = props[0].centroid
    return np.array([int(cy), int(cx)])  # return as (y,x)


In [ ]:
def get_cardinal_points(img,name,grey_value_df):
    """
    Extract both cardinal points from an image.
    Returns dict with 'north' and 'east' as (y,x) arrays.
    """
    points = {}
    for direction in ["north", "east"]:
        grey = grey_value_df.loc[grey_value_df['Mapping_ID'] == direction, 'Tissue_Grey_value'].values[0]
        print(f'Grey value for {direction} is {grey}')
        pt = detect_cardinal_point(img, grey, name)
        if pt is None:
            raise ValueError(f"Could not find '{direction}' point in {name}")
        points[direction] = pt
        print(f"  {direction}: pixel (y,x) ({pt[0]:.1f}, {pt[1]:.1f})")
    return points


In [ ]:
target_spacing = 0.5031 #in um
avg_y = round(float(np.average([5.3,6.1])),4)
avg_x = round(float(np.average([4.1,4.3])),4)
def resize_map(map_arr,target_spacing,avg_y,avg_x):
    """ 
    Resizes image based on target spacing
    Returns the resized image
    """
    map_mask = map_arr < 255
    map_labels = sk.measure.label(map_mask)
    props = sk.measure.regionprops(map_labels)
    map_region = max(props, key=lambda p: p.area)
    area_bbox = map_region.area_bbox
    avg_area = avg_x*avg_y
    spacing_um = np.sqrt(round((avg_area / area_bbox) * 10000**2, 4))
    print(f'calculated spacing: {spacing_um}')
    zoom = spacing_um/target_spacing
    resized_map = sc.ndimage.zoom(map_arr, zoom,order=0)
    return resized_map

In [ ]:
def get_map_region(map_arr, grey_value):
    map_region = (map_arr == int(grey_value)).astype(np.float32)
    map_region = sc.ndimage.binary_fill_holes(map_region)
    return map_region

In [ ]:
def get_region_info(binary_arr,spacing):
    label = sk.measure.label(binary_arr)
    props = sk.measure.regionprops(label)
    print(f'num objects: {len(props)}')
    y, x = props[0].centroid
    min_y, min_x, max_y, max_x = props[0].bbox
    scaled_centroid = [round(float(x*spacing),2),round(float(y*spacing),2)]
    scaled_bbox = [round(float(min_y*spacing),2), round(float(min_x*spacing),2), round(float(max_y*spacing),2), round(float(max_x*spacing),2)]
    return scaled_centroid, scaled_bbox
    

In [ ]:
#update to run through all images in annotation df
def get_metadata(meta_df,grey_value_df,data_path):
    df = meta_df[meta_df['AnimalID'] == animal_id]
    grey_values = []
    img_paths = []
    map_keys = {}
    animal_ids = []
    for _, row in df.iterrows():
        img_path = os.path.join(data_path,row['Tissue.ID'],'cropped_image','cropped_'+row['Image']+'.png')
        img_paths.append(img_path)
        grey_value = grey_value_df.loc[grey_value_df['Mapping_ID'] == row['MappingID'], 'Map_Grey_value'].values[0]
        grey_values.append(grey_value)
        map_key = {row['AnimalID'] : row['MapBase']}
        map_keys.update(map_key)

    return img_paths,grey_values,map_keys



In [ ]:
map_key = np.array(Image.open('Maps\PNGsForReg\GrayscaleMaps\Average4.png').convert('L'))
grey_value_df = pd.read_csv('Maps\PNGsForReg\HexColorMappingKey.2026.2.28.csv',dtype=str,usecols=['Mapping_ID','Map_Grey_value','Tissue_Grey_value'])
df = pd.read_csv('2026.3.4.AnnotationlevelKey_simple.csv',dtype=str)
df = df[df['AnimalID'] == 'SD178']
data_path = 'Pipeline_Test'
map_region_grey_values = []
img_paths = []
names = []
mask_paths = []

for _,row in df.iterrows():
    img_path = os.path.join(data_path,row['Tissue.ID'],'cropped_image','cropped_'+row['Image']+'.png')
    img_paths.append(img_path)
    mask_path = os.path.join(data_path,row['Tissue.ID'],'cropped_mask','cropped_mask_'+row['Image']+'.png')
    mask_paths.append(mask_path)
    grey_value = grey_value_df.loc[grey_value_df['Mapping_ID'] == row['MappingID'], 'Map_Grey_value'].values[0]
    map_region_grey_values.append(grey_value)
    name = row['Tissue.ID']+'_'+row['Image']
    names.append(name)
    

In [ ]:
mask = sk.io.imread(mask_paths[0])
mask.dtype

In [ ]:
viewer = napari.view_image(mask)

In [ ]:
tissue_mask_flip_transforms = []
tissue_mask_rotations = []
tissue_mask_bbox_um = []
for img_path, name, mask_path in zip(img_paths,names,mask_paths):
    img = np.array(Image.open(img_path).convert('L'))
    print(f'img shape: {img.shape}')
    mask = sk.io.imread(mask_path)
    points = get_cardinal_points(img,name,grey_value_df)
    flip_transform, rotation_angle = get_tissue_orientation_transforms(points,mask)
    tissue_mask_flip_transforms.append(flip_transform)
    tissue_mask_rotations.append(rotation_angle)
    _, tissue_bbox = get_region_info(mask,16.1)
    tissue_mask_bbox_um.append(tissue_bbox)
    

In [ ]:
print(tissue_mask_bbox_um)

In [ ]:
tissue_mask_flip_transforms_df = pd.Series(tissue_mask_flip_transforms, name='Flip_Transform', index=df.index)
tissue_mask_rotations_df = pd.Series(tissue_mask_rotations, name='Rotation', index=df.index)
tissue_mask_bbox_um_df = pd.Series(tissue_mask_bbox_um, name='Bounding_box(min_y, min_x, max_y, max_x)',index=df.index)

In [ ]:
#df['Flip_Transform'] = tissue_mask_flip_transforms_df
#df['Rotation'] = tissue_mask_rotations_df
df['Bounding_box(min_y, min_x, max_y, max_x)'] = tissue_mask_bbox_um_df
df.pop('Bounding_box')
df.head()

In [ ]:
fullres_map = resize_map(map_key,target_spacing,avg_y,avg_x)

In [ ]:
map_centroids_um = []
map_bbox_um = []
map_region_ids = []
for value in map_region_grey_values:
    map_region_id = grey_value_df.loc[grey_value_df['Map_Grey_value'] == value, 'Mapping_ID'].values[0]
    map_region_ids.append(map_region_id)
    map_region = get_map_region(fullres_map,value)
    map_centroid, map_bbox = get_region_info(map_region,0.5031)
    map_centroids_um.append(map_centroid)
    map_bbox_um.append(map_bbox)

In [ ]:
map_centroids_um_df = pd.Series(map_centroids_um, name='Map_Region_Centroid', index=df.index)
map_bbox_um_df = pd.Series(map_bbox_um, name='Map_Region_Bounding_box', index=df.index)
map_region_ids_df = pd.Series(map_region_ids, name='Map_Region_ID',index=df.index)
df['Map_Region_Centroid'] = map_centroids_um_df
df['Map_Region_Bounding_box'] = map_bbox_um_df
df['Map_Region_ID'] = map_region_ids_df
df.head()
